Crisis detection part for logistic map

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import math

Constructing the RC skeleton

In [ ]:
class ReservoirComputer:
    
    def __init__(self, dim_system, dim_reservoir, rho, sigma):
        
        self.dim_system = dim_system
        self.dim_reservoir = dim_reservoir
        self.r_state = np.zeros(dim_reservoir)                                   
        self.A = generate_adjacency_matrix(dim_reservoir, rho, sigma)

        
        self.W_in = W_inn(dim_reservoir, dim_system)
        self.W_b = W_bb(dim_reservoir)
        self.W_out = np.zeros((dim_system, dim_reservoir))
        
        

    
    def advance_r_state(self, eps, u):
        alpha = 0.86
        self.r_state = ((1- alpha)* self.r_state) + alpha* np.tanh(np.matmul(self.A, self.r_state) + np.matmul(self.W_in, u.T) + self.W_b * (eps - 0.3))             # r_state is updated and stored as r(t+ delta t)
                                                                                    
        return self.r_state

    
    def train(self, trajectory_total, trajectory_new):                                                # Trajectory is the training data (initialized once model is called)
        
    
        R_total = np.empty((self.dim_reservoir, trajectory_total.shape[0]))
        eps_train = [3.92, 3.93, 3.94, 3.95]

        
        div = int((trajectory_total.shape[0])/len(eps_train))
        
        for m in range(len(eps_train)):

            self.r_state = np.zeros(self.dim_reservoir)
            
            for i in range(div*m , div + div*m):
                
                R_total[:, i] = self.r_state
                u = trajectory_total[i]                                                   
                self.advance_r_state(eps_train[m], u)

        range_removal = [(0,29),(1000,1029),(2000, 2029),(3000, 3029)]
        
        column_removal = []
        for start, end in range_removal:
            column_removal.extend(range(start, end + 1))
        
        # Remove the specified columns
        R_total = np.delete(R_total, column_removal, axis=1)           #removing the r_state transient (first 10 elements)         
        
        odd_column_indices = np.arange(R_total.shape[1])[1::2]         #can be commented out as logistic map doesn't have a symmetry problem (that will slightly change the crisis point in trained RC, since W_out will change)                                                                   

        R_total[:, odd_column_indices] **= 2                           #need to keep only if the exact reproduction of parameter values given in the paper at which critical transition occurs, is desired
        
        self.W_out = linear_regression(R_total, trajectory_new)

        
    def v(self):
        return np.transpose(np.matmul(self.W_out, self.r_state))                   


    def predict(self, eps_new, steps, test_trajectory, train_trajectory):
        
        prediction = np.zeros((steps, self.dim_system))
        prediction[0] = test_trajectory[0,:]

        warmup = train_trajectory[0:30, :]                                         # warmup of reservoir

        self.r_state = np.zeros(self.dim_reservoir)

        for l in range(30):

            self.advance_r_state(eps_new, warmup[l])

        self.r_warmup = self.r_state
        
        for i in range(steps-1):
            
            self.advance_r_state(eps_new, prediction[i])
            v = self.v()
            prediction[i+1] = v
            
        return prediction

    def get_weights(self):
        return self.A, self.W_in, self.W_out, self.W_b, self.r_warmup

In [ ]:
def W_bb(dim_reservoir):

    np.random.seed(36)
    
    W_b = (2 * np.random.rand(dim_reservoir) - 1)* 1.15
    return W_b


def W_inn(dim_reservoir, dim_system):

    np.random.seed(85)
    W_in = 2 * 2.13 * (np.random.rand(dim_reservoir, dim_system) - 0.5)
    return W_in

def generate_adjacency_matrix(dim_reservoir, rho, sigma):
    
    graph = nx.gnp_random_graph(dim_reservoir, sigma, seed = 47)                           # Generates an Erdos-Renyi graph of nodes = dim_reservoir and probab of node connection = sigma
    
    np.random.seed(27)
    nx.set_edge_attributes(graph, {e: {'weight': np.random.uniform(0, 1)} for e in graph.edges})    
    graph = nx.to_numpy_array(graph)

    
    rescaled = graph
    return scale_matrix(rescaled, rho)


def scale_matrix(A, rho):
    eigenvalues, _ = np.linalg.eig(A)
    max_eigenvalue = np.amax(eigenvalues)
    A = (A/np.absolute(max_eigenvalue)) * rho                 
    return A


In [ ]:
def linear_regression(R, trajectory, beta=0.000001):
    Rt = np.transpose(R)
    inverse_part = np.linalg.inv(np.matmul(R, Rt) + beta * np.identity(R.shape[0]))                
    return np.matmul(np.matmul(trajectory.T, Rt), inverse_part)                           

Constructing the data (logistic map time series)

In [ ]:
timesteps = 1100     

mu_list = [3.92, 3.93, 3.94, 3.95, 4.01]      

#mu_list = [3.92, 4.01]
data_list = []

# Initial conditions
for mu in mu_list:
    x0 = 0.3
    for j in range(timesteps):

        x0 = mu*x0*(1-x0)
   
        data_list.append(x0)
   
stacked_total = np.transpose(np.vstack(data_list))                                   
print(stacked_total.shape)

In [ ]:
#train-test split

stacked_train = stacked_total[:, :-1100]
stacked_test = stacked_total[:, -1100:]
stacked_test = stacked_test[:, :1000]        


# removing transient of first 100 points each time
ranges_to_remove = [(0, 99), (1100, 1199), (2200, 2299), (3300,3399)]

#ranges_to_remove = [(0,99)]
columns_to_remove = []
for start, end in ranges_to_remove:
    columns_to_remove.extend(range(start, end + 1))


stacked_x_modified = np.delete(stacked_train, columns_to_remove, axis=1)
                 
training_data = np.transpose(stacked_x_modified)                

# Remove the reservoir transient, 30 points each time
range_to_remove = [(0,29),(1000,1029),(2000, 2029),(3000, 3029)]


column_to_remove = []
for start, end in range_to_remove:
    column_to_remove.extend(range(start, end + 1))

# Remove the specified columns
train_without_transient = np.transpose(np.delete(stacked_x_modified, column_to_remove, axis=1))            

valid_data = np.transpose(stacked_test)              


Checking the RC prediction

In [ ]:
dim_reservoir = 400                                    
model = ReservoirComputer(1, dim_reservoir, 0.9, 0.526)                                                       
model.train(training_data, train_without_transient)
predicted_data = model.predict(4.005 ,len(valid_data), valid_data, train_without_transient)              

In [ ]:
#checking the plots
x_data = valid_data[:,0]

x_pred = predicted_data[:,0]

timeaxis = np.arange(0, (valid_data.shape[0]), 1)
# Plot data on each subplot
h = 20
l = 200
plt.plot(timeaxis[:h], x_data[:h], label = "x")
plt.show()
plt.plot(timeaxis[:l], x_pred[:l], label = "x_pred")
plt.show()


Directional fibers to find out fixed points of the trained RC map

In [ ]:
import numpy as np
import sys
import matplotlib.pyplot as plt
import dfibers.solvers as sv
import dfibers.fixed_points as fx
import dfibers.traversal as tv
import dfibers.numerical_utilities as nu
import dfibers.logging_utilities as lu

# Load model weights
A, W_in, W_out, W_b, rstate_initial = model.get_weights()

Lambda = A + np.matmul(W_in, W_out)
Omega = W_b
alpha = 0.86
N = Lambda.shape[0]

# Fixed-point detection grid (coarse steps)
bif_params_fp = np.arange(3.0, 4.1, 0.025)                                                       

# Bifurcation plot grid (fine steps)
bif_params_plot = np.arange(3.0, 4.1, 0.001)

# Storage for fixed points across bifurcation parameters
all_bif_fixed_points = []
all_bif_fixed_points_proj = []

# Function definitions
def f(V, bif_param):
    return -V + (1 - alpha) * V + alpha * np.tanh(Lambda @ V + Omega[:, None] * (bif_param - 0.3))
    
def Df(V, bif_param):
    N, P = V.shape
    gamma = Lambda @ V + Omega[:, None]*(bif_param - 0.3)
    sech2 = 1 - np.tanh(gamma)**2
    DfV = np.zeros((P, N, N))
    for p in range(P):
        D = np.diag(sech2[:, p])
        DfV[p,:,:] = alpha * (D @ Lambda - np.eye(N))
    return DfV



def ef(V):
    N, P = V.shape
    return 1e-10 * np.ones((N, P))

terminate = lambda trace: np.any(np.abs(trace.x[:N,:]) > 100)
compute_step_amount = lambda trace: (0.001, None, False)

# Duplicates check
duplicates = lambda V, v: (np.abs(V - v) < 1e-5).all(axis=0)

# Holds previous bif_param fixed points
fxpts_prev = []

# Loop over coarse bifurcation parameters
for bif_param in bif_params_fp:
    
    print(f"\nFinding fixed points for bif_param = {bif_param}")

    num_trials = 15
    all_fxpts_this_param = []

    for trial in range(num_trials):

        # Determine initial guess
        if fxpts_prev and trial < len(fxpts_prev):                                                   
            # Use fixed point from previous bif_param run, plus slight perturbation
            v0 = fxpts_prev[trial] + 0.001 * np.random.randn(N, 1)
            
        else:
            v0 = np.random.uniform(-1, 1, size=(N,1))
            

        # Random fiber direction
        c = np.random.randn(N,1)
        c /= np.linalg.norm(c)

        solution = sv.fiber_solver(
            f = lambda V: f(V, bif_param),
            ef = ef,
            Df = lambda V: Df(V, bif_param),
            v = v0,
            c = c,
            terminate = terminate,
            compute_step_amount = compute_step_amount,
            max_traverse_steps = 2000,
            max_solve_iterations = 32,
            logger = None,
        )

        fxpts = solution["Fixed points"]
        
        if fxpts.shape[1] > 0:
            # Sanitize the fixed points from this trial
            sanitized = fx.sanitize_points(
                fxpts,
                f = lambda V: f(V, bif_param),
                ef = ef,
                Df = lambda V: Df(V, bif_param),
                duplicates = duplicates
            )
            if sanitized.shape[1] > 0:
                all_fxpts_this_param.append(sanitized)
            else:
                print(f"  Trial {trial}: all fixed points eliminated during sanitization.")                  #in the above step, duplicates are removed and points with high residue are also removed
        else:
            print(f"  Trial {trial}: No fixed points found.")

    # Combine sanitized points from all trials
    if all_fxpts_this_param:
        fxpts_all = np.hstack(all_fxpts_this_param)

        # Final deduplication
        fxpts_all = fx.sanitize_points(
            fxpts_all,
            f = lambda V: f(V, bif_param),
            ef = ef,
            Df = lambda V: Df(V, bif_param),
            duplicates = duplicates
        )
        

        if fxpts_all.shape[1] > 0:
            print(f"Found {fxpts_all.shape[1]} unique fixed points at bif_param={bif_param}")
        
            stabilities = []  # True = stable, False = unstable
        
            # Residuals and eigenvalues for stability analysis
            for i in range(fxpts_all.shape[1]):
                v_fp = fxpts_all[:, [i]]
                
                # Residual
                residual = np.linalg.norm(f(v_fp, bif_param))
                print(f"  FP {i}: residual = {residual:.3e}")
                
                # Jacobian at this fixed point
                J = Df(v_fp, bif_param)[0,:,:] + np.eye(v_fp.shape[0])                    
                
                # Eigenvalues
                eigvals = np.linalg.eigvals(J)
                max_abs = np.max(np.abs(eigvals))
                print(f"      max |eigenvalue| = {max_abs:.6f}")
        
                stabilities.append(max_abs <= 1.0)                              # Stable if all eigenvalues <= 1 in magnitude
        
            # Project fixed points to output space
    
  
            projections = W_out @ fxpts_all
            all_bif_fixed_points.append(fxpts_all)
            all_bif_fixed_points_proj.append((bif_param, projections, stabilities))
        
            print(projections)
            
            # Save these fixed points as seeds for next bif_param
            fxpts_prev = [fxpts_all[:, [i]] for i in range(fxpts_all.shape[1])]

        else:
            print(f"No unique fixed points survived final sanitization at bif_param={bif_param}")
            fxpts_prev = []
    else:
        print(f"No fixed points found at bif_param={bif_param}")
        fxpts_prev = []




In [ ]:

plt.figure(figsize=(12, 6))

# Separate stable and unstable fixed points
stable_x, stable_y = [], []
unstable_x, unstable_y = [], []

for bif_param, projections, stabilities in all_bif_fixed_points_proj:
    for j in range(projections.shape[1]):
        if stabilities[j]:  # Stable
            stable_x.append(bif_param)
            stable_y.append(projections[0, j])
        else:               # Unstable
            unstable_x.append(bif_param)
            unstable_y.append(projections[0, j])


plt.plot(stable_x, stable_y, 'bx', markersize=5, label="Stable FP")
plt.plot(unstable_x, unstable_y, 'ro', markersize=3, label="Unstable FP")

plt.xlabel("Bifurcation parameter")
plt.ylabel("Fixed points")
plt.legend(loc=(0.6, 0.55), framealpha=0.0)
plt.xlim(3.3, 4.005)
plt.yticks([-15,0, 15])

plt.show()


Predicted bifurcation diagram of trained RC map along with the fixed points

In [ ]:
A, W_in, W_out, W_b, rstate_initial = model.get_weights()

Lambda = A + np.matmul(W_in, W_out)
Omega = W_b


In [ ]:

# Two bifurcation parameter ranges with different step sizes
bif_params_low = np.arange(3.0, 3.5, 0.008)     # coarse
bif_params_high = np.arange(3.5, 4.1, 0.0008)   # fine

# Concatenate them
bif_params_plot = np.concatenate([bif_params_low, bif_params_high])

timestepss = 600
broadcast_shape = 30

bif_repeated = np.repeat(bif_params_plot, broadcast_shape)
bif_broadcasted = bif_repeated.reshape(len(bif_params_plot), broadcast_shape)

x_list = np.empty((1, timestepss))
x_listfinal = np.empty((len(bif_params_plot), broadcast_shape))

for k, bif_param in enumerate(bif_params_plot):
    rstate0 = rstate_initial.copy()
    for j in range(timestepss):
        r = (1-alpha)*rstate0 + alpha*np.tanh(np.matmul(Lambda, rstate0) + Omega*(bif_param - 0.3))
        rstate0 = r
        x_list[:, j] = np.matmul(W_out, r)                                              #Project back to 1D

    x_listfinal[k, :] = x_list[:, -broadcast_shape:]
    x_listfinal[k, :] = x_list[:, -broadcast_shape:]



In [ ]:

plt.figure(figsize=(12, 8))

# Plot bifurcation points of trained RC map
for i in range(len(bif_params_plot)):
    plt.plot(
        bif_broadcasted[i, :],
        x_listfinal[i, :],
        color='black',
        ls='',
        marker=','
    )

# Plot fixed-point projections with stability-based colors
for bif_param, projections, stabilities in all_bif_fixed_points_proj:
    for j in range(projections.shape[1]):
        color = 'blue' if stabilities[j] else 'red'
        plt.plot(
            bif_param,
            projections[0, j],  
            marker='*',
            markersize=5,
            color=color,
            label=(
                "Stable FP" if (color == 'blue' and bif_param == bif_params_fp[0]) else
                "Unstable FP" if (color == 'red' and bif_param == bif_params_fp[0]) else
                None
            )
        )

plt.xlabel("Bifurcation parameter")
plt.ylabel("Output projection")

plt.xlim(3.25, 4.005)
plt.ylim(-0.15, 1.2)

plt.show()
